# Real-data time-lapse resistivity QA and change analysis

This notebook inspects `examples/8_real_data/2_timelapse_inversion_real_adtlert.py` outputs.

Current figures use **raw, temperature-uncorrected** inverted resistivity. The next step is to add the temperature correction used in the WRR paper before hydrologic interpretation.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import PolyCollection
from matplotlib.colors import LogNorm, TwoSlopeNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "axes.labelsize": 13,
        "axes.titlesize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 10,
        "figure.dpi": 120,
    }
)

np.set_printoptions(precision=4, suppress=True)

In [ ]:
# Resolve paths. This works whether the notebook is run from repo root or examples/8_real_data.
here = Path.cwd().resolve()
if (here / "adtlert").exists() and (here / "result").exists():
    project_root = here
else:
    candidates = [here, *here.parents]
    project_root = next(
        (p for p in candidates if (p / "adtlert").exists() and (p / "result").exists()),
        None,
    )
    if project_root is None:
        raise FileNotFoundError(
            "Cannot locate project root containing adtlert/ and result/."
        )

result_dir = (
    project_root / "result" / "8_real_data" / "2_timelapse_inversion_real_adtlert"
)
if not result_dir.exists():
    raise FileNotFoundError(f"Missing result directory: {result_dir}")

save_figures = True
figure_dpi = 600

# Plot controls. Adjust these first if colors are saturated.
resistivity_clim = (50.0, 2200.0)
delta_clim = (-0.75, 0.75)
depth_ylim = (-90.0, 5.0)
upper_depth_m = 1.0
x_bin_count = 80

selected_labels = [
    "2022-03-26 00:30",
    "2022-04-20 06:30",
    "2022-05-02 06:30",
    "2022-05-12 18:30",
]

print("project_root =", project_root)
print("result_dir   =", result_dir)

In [ ]:
# Load inversion arrays and metadata.
summary = json.loads((result_dir / "timelapse_real_inversion_summary.json").read_text())
models = np.load(result_dir / "final_models.npy")
log_models = np.load(result_dir / "final_log_models.npy")
observed_rhoa = np.load(result_dir / "observed_rhoa.npy")
predicted_rhoa = np.load(result_dir / "predicted_rhoa.npy")
coverage = np.load(result_dir / "coverage.npy")
coverage_mask = np.load(result_dir / "coverage_mask.npy").astype(bool)
measurement_times_days = np.load(result_dir / "measurement_times_days.npy")
chi2_all = np.load(result_dir / "chi2_all.npy")

with np.load(result_dir / "timelapse_inversion_mesh.npz") as mesh_data:
    nodes = np.asarray(mesh_data["nodes"], dtype=float)
    cells = np.asarray(mesh_data["cells"], dtype=np.int32)
    elec_x = np.asarray(mesh_data["elec_x"], dtype=float)
    elec_z = np.asarray(mesh_data["elec_z"], dtype=float)
    x_nodes = np.asarray(mesh_data["x_nodes"], dtype=float)
    z_top = np.asarray(mesh_data["z_top"], dtype=float)
    layer_thickness = np.asarray(mesh_data["layer_thickness"], dtype=float)

used_lines = (result_dir / "used_data_files.txt").read_text().splitlines()
time_labels = [line.split("	")[0] for line in used_lines]
data_files = [line.split("	")[1] if "	" in line else line for line in used_lines]
label_to_index = {label: idx for idx, label in enumerate(time_labels)}
selected_indices = [
    label_to_index[label] for label in selected_labels if label in label_to_index
]
if len(selected_indices) != len(selected_labels):
    missing = sorted(set(selected_labels).difference(label_to_index))
    raise ValueError(f"Selected labels not found in used_data_files.txt: {missing}")

cell_centers = nodes[cells].mean(axis=1)
surface_at_center = np.interp(cell_centers[:, 0], x_nodes, z_top)
cell_depth = surface_at_center - cell_centers[:, 1]
visible_cells = (~coverage_mask) & np.isfinite(models[:, 0]) & (cell_depth >= 0.0)

print("n_times:", models.shape[1])
print("n_cells:", models.shape[0])
print("n_data:", observed_rhoa.shape[1])
print("first / last:", time_labels[0], "/", time_labels[-1])
print("selected:", [(i, time_labels[i]) for i in selected_indices])
print("coverage masked cells:", int(coverage_mask.sum()), "/", coverage_mask.size)
summary

## Helper functions


In [ ]:
def add_same_height_colorbar(
    fig, ax, mappable, label=None, title=None, size="4%", pad=0.04
):
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=size, pad=pad)
    cbar = fig.colorbar(mappable, cax=cax)
    if label is not None:
        cbar.set_label(label)
    if title is not None:
        cbar.ax.set_title(title, pad=4)
    cbar.ax.tick_params(labelsize=10)
    return cbar


def format_section_axis(ax, *, show_xlabel=True, show_ylabel=True):
    ax.plot(elec_x, elec_z, color="black", lw=1.1, zorder=5)
    ax.set_xlim(float(elec_x.min()), float(elec_x.max()))
    ax.set_ylim(*depth_ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(False)
    if show_xlabel:
        ax.set_xlabel("Distance (m)")
    else:
        ax.tick_params(axis="x", labelbottom=False)
    if show_ylabel:
        ax.set_ylabel("Relative elevation (m)")
    else:
        ax.tick_params(axis="y", labelleft=False)


def plot_cell_model(ax, values, *, cmap, norm, mask=None, edge=False):
    values = np.asarray(values, dtype=float).ravel()
    visible = np.isfinite(values)
    if mask is not None:
        visible &= ~np.asarray(mask, dtype=bool).ravel()
    pc = PolyCollection(
        nodes[cells][visible],
        array=values[visible],
        cmap=cmap,
        norm=norm,
        edgecolors=(1, 1, 1, 0.18) if edge else "none",
        linewidths=0.15 if edge else 0.0,
    )
    ax.add_collection(pc)
    return pc


def savefig(fig, filename):
    if save_figures:
        out = result_dir / filename
        fig.savefig(out, dpi=figure_dpi, bbox_inches="tight")
        print("saved:", out)

## 1) Data-fit QA

The data-fit metrics are computed in log apparent resistivity space because the inversion objective uses log-domain apparent resistivity residuals.


In [ ]:
obs_log = np.log(observed_rhoa)
pred_log = np.log(predicted_rhoa)
log_residual = pred_log - obs_log
log_rmse = np.sqrt(np.mean(log_residual**2, axis=1))
log_mae = np.mean(np.abs(log_residual), axis=1)
rel_l2_log = np.linalg.norm(log_residual, axis=1) / np.maximum(
    np.linalg.norm(obs_log, axis=1), 1e-12
)

rng = np.random.default_rng(20260528)
all_obs = observed_rhoa.reshape(-1)
all_pred = predicted_rhoa.reshape(-1)
sample_count = min(60000, all_obs.size)
sample_idx = rng.choice(all_obs.size, size=sample_count, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax = axes[0]
ax.scatter(
    all_obs[sample_idx],
    all_pred[sample_idx],
    s=3,
    alpha=0.18,
    color="#2F6DAE",
    rasterized=True,
)
lims = [min(all_obs.min(), all_pred.min()), max(all_obs.max(), all_pred.max())]
ax.plot(lims, lims, color="black", lw=1.2, label="1:1")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Observed apparent resistivity (ohm m)")
ax.set_ylabel("Predicted apparent resistivity (ohm m)")
ax.set_title("Observed vs predicted apparent resistivity")
ax.legend(frameon=True)
ax.grid(alpha=0.25)

ax = axes[1]
ax.plot(measurement_times_days, log_rmse, color="#0072B2", lw=1.9, label="log RMSE")
ax.plot(measurement_times_days, log_mae, color="#E69F00", lw=1.6, label="log MAE")
ax.set_xlabel("Days since baseline")
ax.set_ylabel("Log-domain error")
ax.set_title("Data-fit through the snowmelt window")
ax.grid(alpha=0.25)
ax.legend(frameon=True)

fig.tight_layout()
savefig(fig, "real_timelapse_QA.png")
plt.show()

qa_table = {
    "log_rmse_min": float(log_rmse.min()),
    "log_rmse_max": float(log_rmse.max()),
    "log_rmse_mean": float(log_rmse.mean()),
    "rel_l2_log_mean_percent": float(100.0 * rel_l2_log.mean()),
}
qa_table

## 2) Baseline resistivity model

The first time step (`2022-03-26 00:30`) is used as the baseline model `rho0`.


In [ ]:
baseline_idx = 0
rho0 = models[:, baseline_idx]
rho_norm = LogNorm(vmin=resistivity_clim[0], vmax=resistivity_clim[1])

fig, ax = plt.subplots(figsize=(10.8, 4.2))
pc = plot_cell_model(ax, rho0, cmap="jet", norm=rho_norm, mask=coverage_mask)
format_section_axis(ax)
ax.set_title(
    f"Baseline inverted resistivity (uncorrected): {time_labels[baseline_idx]}"
)
cbar = add_same_height_colorbar(fig, ax, pc, title="rho")
cbar.set_ticks([100, 300, 1000, 2000])
cbar.set_ticklabels(["100", "300", "1000", "2000"])
fig.tight_layout()
savefig(fig, "real_baseline_resistivity.png")
plt.show()

## 3) Relative resistivity change, `Delta rho / rho0`

These panels are the raw, temperature-uncorrected version of the WRR-style time-lapse response. Negative values indicate lower resistivity than the `2022-03-26 00:30` baseline.


In [ ]:
delta_rho_rel = (models - rho0[:, None]) / rho0[:, None]
delta_norm = TwoSlopeNorm(vmin=delta_clim[0], vcenter=0.0, vmax=delta_clim[1])

fig, axes = plt.subplots(
    1, len(selected_indices), figsize=(16.2, 4.2), sharex=True, sharey=True
)
axes = np.atleast_1d(axes)
last_pc = None
for col, (ax, idx) in enumerate(zip(axes, selected_indices, strict=True)):
    last_pc = plot_cell_model(
        ax, delta_rho_rel[:, idx], cmap="RdBu_r", norm=delta_norm, mask=coverage_mask
    )
    format_section_axis(ax, show_xlabel=True, show_ylabel=(col == 0))
    ax.set_title(time_labels[idx])
if last_pc is not None:
    cbar = fig.colorbar(last_pc, ax=axes, fraction=0.018, pad=0.02)
    cbar.set_label("Delta rho / rho0 (uncorrected)")
    cbar.ax.tick_params(labelsize=10)
fig.subplots_adjust(wspace=0.08, right=0.92)
savefig(fig, "real_delta_rho_over_rho0_selected_times.png")
plt.show()

## 4) Uppermost 1 m response along the profile

This approximates the WRR Figure-6 style shallow response by averaging `Delta rho / rho0` over cells whose centers lie within the uppermost 1 m below local topography.


In [ ]:
x_edges = np.linspace(float(elec_x.min()), float(elec_x.max()), x_bin_count + 1)
x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
upper_mask = visible_cells & (cell_depth >= 0.0) & (cell_depth <= upper_depth_m)

upper_profile = np.full((models.shape[1], x_centers.size), np.nan, dtype=float)
for bin_idx in range(x_centers.size):
    in_bin = (
        upper_mask
        & (cell_centers[:, 0] >= x_edges[bin_idx])
        & (cell_centers[:, 0] < x_edges[bin_idx + 1])
    )
    if np.any(in_bin):
        upper_profile[:, bin_idx] = np.nanmean(delta_rho_rel[in_bin, :], axis=0)

fig, axes = plt.subplots(
    2, 1, figsize=(12.0, 7.0), gridspec_kw={"height_ratios": [1.0, 1.1]}
)

ax = axes[0]
line_styles = [
    ("#000000", "-", 2.2),
    ("#0072B2", "-", 1.8),
    ("#D55E00", "--", 1.8),
    ("#009E73", "-.", 1.8),
]
for idx, (color, ls, lw) in zip(selected_indices, line_styles, strict=False):
    ax.plot(
        x_centers, upper_profile[idx], color=color, ls=ls, lw=lw, label=time_labels[idx]
    )
ax.axhline(0.0, color="black", lw=0.9, alpha=0.7)
ax.set_xlim(float(elec_x.min()), float(elec_x.max()))
ax.set_ylabel("Mean Delta rho / rho0")
ax.set_title(f"Uppermost {upper_depth_m:g} m average relative resistivity change")
ax.legend(ncol=2, frameon=True, loc="best")
ax.grid(alpha=0.25)

time_edges = np.empty(measurement_times_days.size + 1, dtype=float)
if measurement_times_days.size > 1:
    time_edges[1:-1] = 0.5 * (measurement_times_days[:-1] + measurement_times_days[1:])
    time_edges[0] = measurement_times_days[0] - (
        time_edges[1] - measurement_times_days[0]
    )
    time_edges[-1] = measurement_times_days[-1] + (
        measurement_times_days[-1] - time_edges[-2]
    )
else:
    time_edges[:] = [measurement_times_days[0] - 0.5, measurement_times_days[0] + 0.5]

ax = axes[1]
mesh = ax.pcolormesh(
    x_edges, time_edges, upper_profile, cmap="RdBu_r", norm=delta_norm, shading="auto"
)
ax.set_xlim(float(elec_x.min()), float(elec_x.max()))
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Days since baseline")
ax.set_title("Time-distance view of shallow relative resistivity change")
cbar = fig.colorbar(mesh, ax=ax, fraction=0.020, pad=0.015)
cbar.set_label("Delta rho / rho0 (uncorrected)")

fig.tight_layout()
savefig(fig, "real_upper1m_delta_rho_over_rho0.png")
plt.show()

print("Upper-depth cells used:", int(upper_mask.sum()))

## Notes for the next step

If the raw `Delta rho / rho0` patterns are hydrologically plausible, the next notebook/script should add the WRR-style temperature correction before converting the ERT response into water-content change.
